# **VECHILE SOUND DETECTION PROJECT USING LSTM(RNN)**

In [10]:
## 🛠️ Project Roadmap

### 1. Setup Environment

!pip install datasets librosa tensorflow

In [5]:
### 2. Load Dataset

from datasets import load_dataset

# Load the only available split
dataset = load_dataset("DynamicSuperb/Vehicle_sounds_classification_dataset", split="test")

# Manually split into train/validation
dataset = dataset.train_test_split(test_size=0.2, seed=42)

train_ds = dataset["train"]
val_ds = dataset["test"]

print("Train size:", len(train_ds))
print("Validation size:", len(val_ds))
print(train_ds[0])  # Inspect first sample



Train size: 1364
Validation size: 341
{'audio': <datasets.features._torchcodec.AudioDecoder object at 0x00000266EC825FD0>, 'file': 'VehicleNoise1492', 'instruction': 'Listen to the audio, what class of vehicle sounds does it belong to? The answer could be 0 (car class), 1 (truck, bus and van class), 2 (motorcycle class).', 'label': 0}


In [8]:
### 3. Preprocess Audio → Spectrograms
#RNNs work best with sequential data. We’ll convert audio into **MFCC features** (time-series representation).


import librosa
import numpy as np

def extract_features(batch, n_mfcc=40, max_len=200):
    audio_array = np.array(batch["audio"]["array"])  # waveform
    sr = batch["audio"]["sampling_rate"]             # sample rate

    # Compute MFCCs
    mfccs = librosa.feature.mfcc(y=audio_array, sr=sr, n_mfcc=n_mfcc)

    # Pad or truncate to fixed length
    if mfccs.shape[1] < max_len:
        pad_width = max_len - mfccs.shape[1]
        mfccs = np.pad(mfccs, ((0,0),(0,pad_width)), mode="constant")
    else:
        mfccs = mfccs[:, :max_len]

    batch["features"] = mfccs.T.astype(np.float32)  # shape (time, features)
    return batch

dataset = dataset.map(extract_features)

Map:   0%|          | 0/1364 [00:00<?, ? examples/s]

Map:   0%|          | 0/341 [00:00<?, ? examples/s]

In [20]:
### 4. Prepare Data for Training

#Pad sequences so they fit into batches.


from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Select the training split
train_ds = dataset["train"]

# ✅ Fit encoder on text labels (e.g., "car_horn", "truck_engine")
encoder = LabelEncoder()
encoder.fit(train_ds["label"])   # <-- replace your old numeric fit here

# Transform labels into integers
labels = encoder.transform(train_ds["label"])

# Collect features into a list
features = [np.array(f) for f in train_ds["features"]]

# Pad sequences
X = pad_sequences(features, padding="post", dtype="float32")
y = labels

print(X.shape, y.shape)




(1364, 200, 40) (1364,)


In [21]:
### 5. Build RNN Model
#We’ll use **LSTM layers** for sequential learning.


import tensorflow as tf
from tensorflow.keras import layers, models

# Build RNN with LSTM
model = models.Sequential([
    # Mask padded values (important for variable-length sequences)
    layers.Masking(mask_value=0., input_shape=(X.shape[1], X.shape[2])),

    # First LSTM layer
    layers.LSTM(128, return_sequences=True),
    layers.Dropout(0.3),

    # Second LSTM layer
    layers.LSTM(64),
    layers.Dropout(0.3),

    # Dense layers for classification
    layers.Dense(64, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    # Output layer
    layers.Dense(len(encoder.classes_), activation="softmax")
])

# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


C:\Users\LENOVO\anaconda3\Lib\site-packages\keras\src\layers\core\masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
C:\Users\LENOVO\anaconda3\Lib\site-packages\keras\src\layers\layer.py:1072: UserWarning: Layer 'lstm_4' (of type LSTM) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ masking_2 (Masking)                  │ (None, 200, 40)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_4 (LSTM)                        │ (None, 200, 128)            │          86,528 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 200, 128)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_5 (LSTM)                        │ (None, 64)                  │          49,408 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_4 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 64)                  │           4,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 64)                  │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_5 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 3)                   │             195 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 140,547 (549.01 KB)

 Trainable params: 140,419 (548.51 KB)

 Non-trainable params: 128 (512.00 B)

In [22]:
### 6. Train Model

history = model.fit(X, y, epochs=20, batch_size=32, validation_split=0.2)

Epoch 1/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 7s 160ms/step - accuracy: 0.4189 - loss: 1.3498 - val_accuracy: 0.7582 - val_loss: 0.8549
Epoch 2/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 4s 102ms/step - accuracy: 0.5976 - loss: 0.9711 - val_accuracy: 0.7619 - val_loss: 0.7309
Epoch 3/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 4s 103ms/step - accuracy: 0.6984 - loss: 0.7898 - val_accuracy: 0.7912 - val_loss: 0.6429
Epoch 4/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 4s 104ms/step - accuracy: 0.7168 - loss: 0.7718 - val_accuracy: 0.7949 - val_loss: 0.5691
Epoch 5/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 4s 104ms/step - accuracy: 0.7635 - loss: 0.6583 - val_accuracy: 0.8059 - val_loss: 0.5168
Epoch 6/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 3s 98ms/step - accuracy: 0.7809 - loss: 0.6361 - val_accuracy: 0.8205 - val_loss: 0.4677
Epoch 7/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 3s 91ms/step - accuracy: 0.8093 - loss: 0.5746 - val_accuracy: 0.8425 - val_loss: 0.4624
Epoch 8/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 3s 93ms/step - accuracy: 0.8011 - loss: 0.5462 - val_accuracy: 0.860

In [23]:
### 7. Evaluate

loss, acc = model.evaluate(X, y)
print(f"Accuracy: {acc:.2f}")

43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.8394 - loss: 0.4775
Accuracy: 0.84


In [24]:
### 8. Save & Deploy

model.save("vehicle_sound_rnn.h5")

#You can later load it for inference:

loaded_model = tf.keras.models.load_model("vehicle_sound_rnn.h5")

# testing process

In [26]:
### 🔎 Step 1: Load a test audio file

import librosa
import numpy as np
import tensorflow as tf

# Load your trained model
loaded_model = tf.keras.models.load_model("vehicle_sound_rnn.h5")

# Example: user-provided audio file
file_path = "Downloads/audio/audio3.wav"  # replace with your own .wav file

### 🎼 Step 2: Preprocess audio (same as training)

def preprocess_audio(file_path, n_mfcc=40, max_len=200):
    audio, sr = librosa.load(file_path, sr=22050)
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)

    # Pad or truncate
    if mfccs.shape[1] < max_len:
        pad_width = max_len - mfccs.shape[1]
        mfccs = np.pad(mfccs, ((0,0),(0,pad_width)), mode="constant")
    else:
        mfccs = mfccs[:, :max_len]

    return mfccs.T[np.newaxis, ...]  # shape (1, time, features)

    ### 🧠 Step 3: Run prediction

# Preprocess the audio
X_test = preprocess_audio(file_path)

pred = loaded_model.predict(X_test)
pred_class = pred.argmax(axis=1)[0]

# Convert numeric class to text label
pred_label = encoder.classes_[pred_class]
# Listen  
# 0 (car class), 
# 1 (truck, bus and van class),
# 2 (motorcycle class).

print("Predicted class index:", pred_class)
print("Predicted class label:", pred_label)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 256ms/step
Predicted class index: 0
Predicted class label: 0
